# einsum-contraction — ex3: trilinear interpolation as a three-tensor einsum

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einsum-contraction`. Running the final beacon cell reports progress against the `Einsum: Index contraction semantics` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einsum: Index contraction semantics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einsum-contraction`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einsum-contraction"
DD_SUBTOPIC = "Einsum: Index contraction semantics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einsum index contraction — quick refresher

In `einsum('...->...', *tensors)`, an index that appears on the input side but NOT on the output side is **contracted** (summed). An index that appears on both sides is preserved as a free axis. Repeated inputs of the same letter across different operands cause that letter to be **matched then contracted** — this is how multi-tensor products (trilinear interp, factor models) collapse cleanly to a single line.

### Exercise 3 — trilinear interpolation as a three-tensor einsum

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Compose a three-operand einsum that contracts three separate 1-D weight axes simultaneously against a 3-D voxel grid, implementing factorised trilinear interpolation in one line.
> Keywords: trilinear, multi-tensor-einsum, factorised-weights, nerf-volumetric
> ```

**KCs targeted:** `einsum-repeated-index-sums`, `einsum-batch-axis-passthrough`

ex1 and ex2 covered single-axis contraction (`'bij,i->bj'`-style). This drill is a *three-operand* contraction — factorised trilinear interpolation, the kind used in tri-plane NeRFs and volumetric grids.

Implement `ex3_trilinear(vol, wx, wy, wz)`:

1. `vol` is a voxel grid `(D, H, W)` — values to interpolate.
2. `wx`, `wy`, `wz` are 1-D weight vectors of length `W`, `H`, `D` respectively, each summing to 1.
3. Compute the scalar `sum_{d,h,w} vol[d,h,w] * wz[d] * wy[h] * wx[w]` using a SINGLE `t.einsum` call.
4. The einsum pattern should treat `d`, `h`, `w` as repeated indices contracted across `vol` and the three weight vectors.

Hint: the pattern looks like `'dhw,w,h,d->'` (all four indices contract).

Output: scalar `float32` tensor (shape `()`).

In [ ]:
def ex3_trilinear(vol: Tensor, wx: Tensor, wy: Tensor, wz: Tensor) -> Tensor:
    return t.einsum('dhw,w,h,d->', vol, wx, wy, wz).to(t.float32)


<details><summary>Solution</summary>

```python
def ex3_trilinear(vol: Tensor, wx: Tensor, wy: Tensor, wz: Tensor) -> Tensor:
    return t.einsum('dhw,w,h,d->', vol, wx, wy, wz).to(t.float32)
```

**Multi-operand contraction in one symbol per axis.** Each letter `d`, `h`, `w` appears in `vol` AND in one weight vector AND is absent from the output → it gets matched across operands then summed. einsum handles the broadcast-and-reduce in one shot; you don't need three separate `*` and `.sum()` calls.

**Why this generalises to NeRFs.** Tri-plane and factorised-grid models store the volume as outer products of low-rank axis tensors. Sampling is exactly this contraction — `einsum` makes it 10× more readable than nested `unsqueeze` + `*` + `sum`.

**`'dhw,w,h,d->'` vs `'dhw,d,h,w->'`.** Order of operands doesn't matter as long as each letter shows up in exactly one of the weight inputs. We listed `wx, wy, wz` to match spatial reading order; einsum reorders internally.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()